# YOLOv8s — DUO transfer (Brackish pretrained)

**Model B** в Transfer Learning A/B на датасете DUO.

- Стартовые веса: чекпоинт YOLOv8s, предобученной на Brackish (см. переменную `BRACKISH_WEIGHTS` ниже).
- Fine-tune на DUO (4 класса: holothurian, echinus, scallop, starfish).
- **Все гиперпараметры идентичны `train_duo_baseline.ipynb`** — это требование A/B: единственная разница должна быть в источнике предобучения.

Гипотеза: предобучение на близком домене (Brackish underwater) даст лучший fine-tune на DUO, чем общее COCO.

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Working dir: {Path.cwd()}")

import torch
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:        {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Путь к Brackish-предобученным весам

Подставь путь к `best.pt` от своего лучшего прогона v8s на Brackish (например, `runs/yolov8s/weak/weights/best.pt`).
Скрипт упадёт с понятной ошибкой, если файл не найден — это защита от того, что случайно стартанёшь с дефолтных COCO весов.

In [ ]:
BRACKISH_WEIGHTS = Path("runs/yolov8s/weak/weights/best.pt")

assert BRACKISH_WEIGHTS.exists(), (
    f"Не найден чекпоинт Brackish-предобучения: {BRACKISH_WEIGHTS}\n"
    "Поправь переменную BRACKISH_WEIGHTS на путь к best.pt от своего прогона v8s на Brackish."
)
print(f"Brackish weights: {BRACKISH_WEIGHTS}  ({BRACKISH_WEIGHTS.stat().st_size / 1e6:.1f} MB)")

In [ ]:
model = YOLO(str(BRACKISH_WEIGHTS))

model.train(
    data="configs/dataset_duo.yaml",
    epochs=30,
    imgsz=640,
    batch=32,
    patience=10,
    device=0,
    workers=4,
    seed=42,
    amp=True,
    # weak augmentation — идентично baseline
    fliplr=0.5,
    hsv_h=0.014,
    hsv_s=0.1,
    hsv_v=0.1,
    degrees=0,
    translate=0.0,
    scale=0.0,
    # output
    project="runs/duo_yolov8s",
    name="transfer",
)

## Training curves

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

candidates = sorted(
    Path("runs/duo_yolov8s").glob("transfer*/results.csv"),
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError("results.csv не найден — возможно, обучение упало.")
RESULTS_CSV = candidates[-1]
print(f"Reading: {RESULTS_CSV}")

df = pd.read_csv(RESULTS_CSV, skipinitialspace=True)

metrics = {
    "mAP@50":    "metrics/mAP50(B)",
    "mAP@50-95": "metrics/mAP50-95(B)",
    "Precision": "metrics/precision(B)",
    "Recall":    "metrics/recall(B)",
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (title, col) in zip(axes, metrics.items()):
    ax.plot(df[col], linewidth=2)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

plt.suptitle("YOLOv8s — DUO transfer (Brackish pretrained)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nBest mAP@50:    {df[metrics['mAP@50']].max():.4f}  (epoch {df[metrics['mAP@50']].argmax() + 1})")
print(f"Best mAP@50-95: {df[metrics['mAP@50-95']].max():.4f}  (epoch {df[metrics['mAP@50-95']].argmax() + 1})")

## Evaluate on DUO test set

In [ ]:
weights_candidates = sorted(
    Path("runs/duo_yolov8s").glob("transfer*/weights/best.pt"),
    key=lambda p: p.stat().st_mtime,
)
WEIGHTS = weights_candidates[-1]
print(f"Using: {WEIGHTS}")

best = YOLO(str(WEIGHTS))

metrics = best.val(
    data="configs/dataset_duo.yaml",
    split="test",
    batch=1,
    device=0,
    plots=True,
)

print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

print(f"\nInference speed: {metrics.speed}")

print("\nPer-class AP@50:")
for i, name in metrics.names.items():
    print(f"  {name:12s}  {metrics.box.ap50[i]:.4f}")